# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [5]:
import pandas as pd

# import the data set
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

print("=" * 60)
print("DATASET ORIGINAL")
print("=" * 60)
print(f"Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")

# 2. clean the columns
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

print(f"clean columns:\n{df.columns.tolist()}\n")

# check the data
print("null values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f"values in 'response':\n{df['response'].value_counts()}")
print(f"summarize of'total_claim_amount':\n{df['total_claim_amount'].describe()}\n")

# create the filter data frame
CLAIM_THRESHOLD = 1000

mask_low_claim    = df["total_claim_amount"] < CLAIM_THRESHOLD
mask_yes_response = df["response"].str.strip().str.title() == "Yes"

filtered_df = df[mask_low_claim & mask_yes_response].copy()

print("=" * 60)
print("filter data set")
print("=" * 60)
print(f"rows: {len(filtered_df):,} ({len(filtered_df)/len(df)*100:.1f}% del total)")
print(f"Shape: {filtered_df.shape}")
print(f"  total_claim_amount < ${CLAIM_THRESHOLD:,}")
print(f"  response == 'Yes'")

print("\nEstadisticas de total_claim_amount (filtrado):")
print(filtered_df["total_claim_amount"].describe().round(2))

print("\nTipos de poliza mas frecuentes:")
print(filtered_df["policy_type"].value_counts())

print("\nDistribucion por genero:")
print(filtered_df["gender"].value_counts())

print("\nPrimeras 10 filas:")
cols_preview = ["customer", "response", "total_claim_amount", "policy_type", "gender", "income"]
print(filtered_df[cols_preview].head(10).to_string(index=False))

# 6. export the result
output_path = "filtered_customers.csv"
filtered_df.to_csv(output_path, index=False)
print(f"data frame: {output_path}")

DATASET ORIGINAL
Shape: 10,910 filas x 26 columnas
clean columns:
['unnamed:_0', 'customer', 'state', 'customer_lifetime_value', 'response', 'coverage', 'education', 'effective_to_date', 'employmentstatus', 'gender', 'income', 'location_code', 'marital_status', 'monthly_premium_auto', 'months_since_last_claim', 'months_since_policy_inception', 'number_of_open_complaints', 'number_of_policies', 'policy_type', 'policy', 'renew_offer_type', 'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size', 'vehicle_type']

null values:
state                         631
response                      631
months_since_last_claim       633
number_of_open_complaints     633
vehicle_class                 622
vehicle_size                  622
vehicle_type                 5482
dtype: int64
values in 'response':
response
No     8813
Yes    1466
Name: count, dtype: int64
summarize of'total_claim_amount':
count    10910.000000
mean       434.888330
std        292.180556
min          0.099007
25

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [8]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Filter the yes answer
yes_df = df[df["response"].str.strip().str.title() == "Yes"].copy()

# Combined summary table
summary = (
    yes_df
    .groupby(["policy_type", "gender"])
    .agg(
        avg_monthly_premium = ("monthly_premium_auto", "mean"),
        avg_clv             = ("customer_lifetime_value", "mean"),
        avg_claim           = ("total_claim_amount", "mean"),
        customer_count      = ("customer_lifetime_value", "count")
    )
    .round(2)
    .reset_index()
)
summary["clv_to_claim_ratio"] = (summary["avg_clv"] / summary["avg_claim"]).round(2)

print(summary.to_string(index=False))

print("\nTop segments by profitability (CLV-to-claim ratio):")
top = summary.sort_values("clv_to_claim_ratio", ascending=False)
print(top[["policy_type", "gender", "avg_clv", "avg_claim", "clv_to_claim_ratio"]].to_string(index=False))

   policy_type gender  avg_monthly_premium  avg_clv  avg_claim  customer_count  clv_to_claim_ratio
Corporate Auto      F                94.30  7712.63     433.74             169               17.78
Corporate Auto      M                92.19  7944.47     408.58             154               19.44
 Personal Auto      F                99.00  8339.79     452.97             540               18.41
 Personal Auto      M                91.09  7448.38     457.01             536               16.30
  Special Auto      F                92.31  7691.58     453.28              35               16.97
  Special Auto      M                86.34  8247.09     429.53              32               19.20

Top segments by profitability (CLV-to-claim ratio):
   policy_type gender  avg_clv  avg_claim  clv_to_claim_ratio
Corporate Auto      M  7944.47     408.58               19.44
  Special Auto      M  8247.09     429.53               19.20
 Personal Auto      F  8339.79     452.97               18.41
Corpor

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean columns
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Count customers per state
customers_by_state = (
    df.groupby("state")["customer"]
    .count()
    .reset_index()
    .rename(columns={"customer": "customer_count"})
    .sort_values("customer_count", ascending=False)
)

print("Customers by state (all):")
print(customers_by_state.to_string(index=False))

# states with more than 500 customers
states_over_500 = customers_by_state[customers_by_state["customer_count"] > 500]

print("\nStates with more than 500 customers:")
print(states_over_500.to_string(index=False))

Customers by state (all):
     state  customer_count
California            3552
    Oregon            2909
   Arizona            1937
    Nevada             993
Washington             888

States with more than 500 customers:
     state  customer_count
California            3552
    Oregon            2909
   Arizona            1937
    Nevada             993
Washington             888


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [10]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Max, min, median CLV by education and gender
clv_summary = (
    df.groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(
        max_clv    = "max",
        min_clv    = "min",
        median_clv = "median"
    )
    .round(2)
    .reset_index()
)

print(clv_summary.to_string(index=False))

           education gender  max_clv  min_clv  median_clv
            Bachelor      F 73225.96  1904.00     5640.51
            Bachelor      M 67907.27  1898.01     5548.03
             College      F 61850.19  1898.68     5623.61
             College      M 61134.68  1918.12     6005.85
              Doctor      F 44856.11  2395.57     5332.46
              Doctor      M 32677.34  2267.60     5577.67
High School or Below      F 55277.45  2144.92     6039.55
High School or Below      M 83325.38  1940.98     6286.73
              Master      F 51016.07  2417.78     5729.86
              Master      M 50568.26  2272.31     5579.10


In [ ]:
#"education and gender alone dont fully explain CLV , other factors like policy type and tenure also play a major role
#the max_clv for doctor-educated customers tends to be among the highest across both genders
#typical customer behaves similarly regardless of education or gende


## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [11]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Extract month from effective_to_date
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])
df["month"] = df["effective_to_date"].dt.month_name()

# Pivot table
pivot = (
    df.pivot_table(
        index="state",
        columns="month",
        values="customer",
        aggfunc="count",
        fill_value=0
    )
)

# Order columns by calendar month
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]
pivot = pivot.reindex(columns=[m for m in month_order if m in pivot.columns])

# Add a total column
pivot["total"] = pivot.sum(axis=1)

print(pivot.to_string())

month       January  February  total
state                               
Arizona        1008       929   1937
California     1918      1634   3552
Nevada          551       442    993
Oregon         1565      1344   2909
Washington      463       425    888


C:\Users\punk_\AppData\Local\Temp\ipykernel_28580\1090459528.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])


6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [13]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

# Extract month
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])
df["month"] = df["effective_to_date"].dt.month_name()

# Step 1: Group by state and month, count policies
policies_by_state_month = (
    df.groupby(["state", "month"])["customer"]
    .count()
    .reset_index()
    .rename(columns={"customer": "policies_sold"})
)

# Step 2: Get total policies per state and select top 3
top3_states = (
    policies_by_state_month
    .groupby("state")["policies_sold"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
    .index
    .tolist()
)

print("Top 3 states:", top3_states)

# Step 3: Filter to top 3 states only
top3_df = policies_by_state_month[policies_by_state_month["state"].isin(top3_states)].copy()

# Order months chronologically
month_order = ["January", "February", "March", "April", "May", "June",
               "July", "August", "September", "October", "November", "December"]
top3_df["month"] = pd.Categorical(top3_df["month"], categories=month_order, ordered=True)
top3_df = top3_df.sort_values(["state", "month"]).reset_index(drop=True)

print("\nPolicies sold by month for top 3 states:")
print(top3_df.to_string(index=False))


Top 3 states: ['California', 'Oregon', 'Arizona']

Policies sold by month for top 3 states:
     state    month  policies_sold
   Arizona  January           1008
   Arizona February            929
California  January           1918
California February           1634
    Oregon  January           1565
    Oregon February           1344


C:\Users\punk_\AppData\Local\Temp\ipykernel_28580\718437459.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["effective_to_date"] = pd.to_datetime(df["effective_to_date"])


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [14]:
#import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

print("Sales channels available:")
print(df["sales_channel"].value_counts())
print("\nResponse values:")
print(df["response"].value_counts())

# create binary response column
df["responded_yes"] = (df["response"].str.strip().str.title() == "Yes").astype(int)

# Step 2: Melt to long format — one row per customer per channel type
channel_cols = ["customer", "responded_yes", "sales_channel", "renew_offer_type", "vehicle_class"]

melted = df[channel_cols].melt(
    id_vars       = ["customer", "responded_yes"],
    value_vars    = ["sales_channel", "renew_offer_type", "vehicle_class"],
    var_name      = "channel_type",
    value_name    = "channel_value"
)

# Step 3: Compute response rate per channel type and value
response_rate = (
    melted
    .groupby(["channel_type", "channel_value"])
    .agg(
        total_customers = ("customer", "count"),
        total_yes       = ("responded_yes", "sum")
    )
    .reset_index()
)

response_rate["response_rate_%"] = (
    (response_rate["total_yes"] / response_rate["total_customers"]) * 100
).round(2)

response_rate = response_rate.sort_values(
    ["channel_type", "response_rate_%"], ascending=[True, False]
).reset_index(drop=True)

print("\nResponse rate by marketing channel:")
print(response_rate.to_string(index=False))

Sales channels available:
sales_channel
Agent          4121
Branch         3022
Call Center    2141
Web            1626
Name: count, dtype: int64

Response values:
response
No     8813
Yes    1466
Name: count, dtype: int64

Response rate by marketing channel:
    channel_type channel_value  total_customers  total_yes  response_rate_%
renew_offer_type        Offer2             3490        769            22.03
renew_offer_type        Offer1             4483        665            14.83
renew_offer_type        Offer3             1732         32             1.85
renew_offer_type        Offer4             1205          0             0.00
   sales_channel         Agent             4121        742            18.01
   sales_channel           Web             1626        177            10.89
   sales_channel        Branch             3022        326            10.79
   sales_channel   Call Center             2141        221            10.32
   vehicle_class    Sports Car              550         